# ToolCorrectnessMetric

## What it measures

Whether the agent called the tools it was supposed to call. It compares `tools_called`
against `expected_tools` and scores the overlap. Optionally it also compares each call's
`input_parameters` and the ordering.

**This is the only metric in the suite that scores without an LLM judge.** Scoring is
set comparison, so it is deterministic, free, and fast - which makes it the right
regression gate for tool-calling behaviour.

Note one caveat specific to the installed version: in DeepEval 4.1.4
`ToolCorrectnessMetric.__init__` still constructs a `GPTModel`, which raises
`DeepEvalError: OpenAI API key is not configured` when `OPENAI_API_KEY` is absent. No LLM
call is made when the metric runs, but the key must be present for the constructor to
succeed. This notebook therefore still requires `OPENAI_API_KEY` to be set, even though
none of the judging is done by a model.

## When it is useful

As the primary guard on an agent's planning layer. Tool selection is where an agent
silently degrades: a prompt change or a model swap starts skipping the sanctions screen,
the final answer still reads plausibly, and nothing else notices. A deterministic
comparison against an expected plan catches that on the first run.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `LLMTestCase` |
| `input` | yes |
| `tools_called` | yes - `list[ToolCall]` |
| `expected_tools` | yes - `list[ToolCall]` |

`ToolCall` carries `name`, `input_parameters` and `output`. Which of those participate in
scoring is controlled by the metric's `evaluation_params`.

In [ ]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

In [ ]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
EXPECTED_SCHEMA_VERSION = "1.0.0"

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    if served and served != EXPECTED_SCHEMA_VERSION:
        print(f"WARNING: application reports contract version {served}, these "
              f"notebooks were written against {EXPECTED_SCHEMA_VERSION}. "
              f"Field names may have changed - see docs/evaluation-contract.md.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

## Endpoints exercised

| Endpoint | Role here |
|---|---|
| `POST /api/cases/{case_id}/investigate` | Runs the agent and produces the actual tool calls |
| `GET /api/eval/export/{run_id}` | Returns the same run with `tools_called` already in DeepEval's `ToolCall` field names (`name`, `input_parameters`, `output`) |
| `GET /api/mcp/tools` | The live tool catalogue with each tool's own JSON Schema |
| `GET /api/cases/{id}`, `/api/customers/{id}`, `/api/beneficiaries/{id}` | The facts the expected plan is computed from |

Using the eval export rather than the investigation response for `tools_called` is
deliberate: the export already namespaces each call as `<server>.<tool>` and uses
`input_parameters` rather than `input`, so no field renaming happens in this notebook and
there is nothing for a mapping bug to hide behind.

**Reset matters for this notebook.** The planner skips any tool that already succeeded for
this case in an earlier run, so the expected plan is only valid on the first investigation
after a reset. Set `AML_RESET_BEFORE_RUN=true`.

In [ ]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

if not RESET_BEFORE_RUN:
    print()
    print("NOTE: AML_RESET_BEFORE_RUN is false. The tool planner skips any tool that "
          "already ran successfully\n      for this case in an earlier run, so the "
          "expected plan below is only valid on the\n      first investigation of this "
          "case after a reset. If this notebook reports missing\n      tools, set "
          "AML_RESET_BEFORE_RUN=true and re-run.")

In [ ]:
# --------------------------------------------------------------------------
# The exact request.
#
# This endpoint reads no request body: the case_id in the path is the entire
# input and all context is loaded server-side.
# --------------------------------------------------------------------------
CASE_ID = SCENARIOS["s3"]["case_id"]   # "Possible sanctions name match (beneficiary near-miss)"

print("POST", f"{API_BASE}/api/cases/{CASE_ID}/investigate")
print("headers:", json.dumps(redact({"X-API-Key": API_KEYS["analyst"] or None,
                                     "Content-Type": "application/json"}), indent=2))
print("body: (none)")

In [ ]:
# --------------------------------------------------------------------------
# The raw responses: the investigation, then the same run reshaped into
# DeepEval's LLMTestCase field names by GET /api/eval/export/{run_id}.
# --------------------------------------------------------------------------
investigation = api("POST", f"/api/cases/{CASE_ID}/investigate", expect_status=201)
RUN_ID = investigation["run_id"]
print(f"run_id = {RUN_ID}\n")

export = api("GET", f"/api/eval/export/{RUN_ID}", role="eval_reader")

show("GET /api/eval/export/{run_id}",
     {k: v for k, v in export.items() if k not in ("retrieval_context", "metadata")})
print()
print("tools_called, in DeepEval ToolCall field names, straight from the API:")
for entry in export["tools_called"]:
    print(f"  name             : {entry['name']}")
    print(f"  input_parameters : {json.dumps(entry['input_parameters'])}")
    print(f"  output           : {json.dumps(entry['output'])[:180]}")
    print()

## Mapping the API response onto DeepEval fields

| DeepEval field | API field | Note |
|---|---|---|
| `input` | `input` from the eval export | A deterministic statement of the investigation task |
| `actual_output` | `actual_output` | Not scored by this metric; kept so the case is well-formed |
| `tools_called` | `tools_called[]` | One-to-one: `name` -> `name`, `input_parameters` -> `input_parameters`, `output` -> `output` |
| `expected_tools` | derived below | Computed from case facts and the documented planner rules |

The export sets `expected_tools: null` permanently and by design - the application refuses
to serve its own answer key, on the grounds that an application supplying its own goldens
would be a participant in its own grading. The harness supplies it.

In [ ]:
# --------------------------------------------------------------------------
# Deriving the expected tool plan.
#
# The application's tool planner is deterministic and rule-based, and the rules
# are documented. Every input those rules need is reachable black-box:
#
#   GET /api/cases/{id}          -> whether a beneficiary exists
#   GET /api/customers/{id}      -> type, country, incorporation_or_dob
#   GET /api/beneficiaries/{id}  -> country, identifiers.dob
#   GET /api/mcp/tools           -> the tool namespace and each input schema
#
# The rules, as published:
#   1. Screen the customer for sanctions/PEP, and the beneficiary too if there
#      is one. A dob is included only for an individual customer, and for a
#      beneficiary only when identifiers.dob is an ISO date.
#   2. Always search adverse media for the customer.
#   3. Check the beneficiary's country for high risk - skipped when there is no
#      beneficiary, or when the beneficiary country equals the customer country
#      (no cross-border exposure).
#   4. Fetch an external registry extract only for a business customer.
#
# Nothing here is copied from the run under test; the plan is computed from
# case facts before the actual calls are looked at.
# --------------------------------------------------------------------------
import re

from deepeval.test_case import ToolCall

case = api("GET", f"/api/cases/{CASE_ID}")
customer = api("GET", f"/api/customers/{case['customer_id']}")
beneficiary = None
if case.get("transaction") and case["transaction"].get("beneficiary_id"):
    beneficiary = api("GET", f"/api/beneficiaries/{case['transaction']['beneficiary_id']}")

print("planner inputs")
print(f"  customer    : name={customer['name']!r} type={customer['type']!r} "
      f"country={customer['country']!r} dob_or_inc={customer.get('incorporation_or_dob')!r}")
print(f"  beneficiary : {beneficiary and beneficiary['name']!r} "
      f"country={beneficiary and beneficiary['country']!r} "
      f"identifiers={beneficiary and beneficiary.get('identifiers')}")
print()

ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}$")

expected_tools = []

# Rule 1 - sanctions/PEP screening.
customer_args = {"name": customer["name"]}
if customer["type"] == "individual" and ISO_DATE.match(
        str(customer.get("incorporation_or_dob") or "")):
    customer_args["dob"] = customer["incorporation_or_dob"]
expected_tools.append(ToolCall(name="risk_screening.screen_sanctions_pep",
                               input_parameters=customer_args))

if beneficiary is not None:
    beneficiary_args = {"name": beneficiary["name"]}
    dob = (beneficiary.get("identifiers") or {}).get("dob")
    if dob and ISO_DATE.match(str(dob)):
        beneficiary_args["dob"] = dob
    expected_tools.append(ToolCall(name="risk_screening.screen_sanctions_pep",
                                   input_parameters=beneficiary_args))

# Rule 2 - adverse media, always, for the customer.
expected_tools.append(ToolCall(name="risk_screening.search_adverse_media",
                               input_parameters={"name": customer["name"]}))

# Rule 3 - high-risk country, only for cross-border exposure.
if beneficiary is not None and beneficiary["country"] != customer["country"]:
    expected_tools.append(ToolCall(name="risk_screening.check_high_risk_country",
                                   input_parameters={"country_code": beneficiary["country"]}))
else:
    print("rule 3 not applicable: no beneficiary, or beneficiary country equals "
          "customer country (no cross-border exposure)")

# Rule 4 - external registry evidence, business customers only.
if customer["type"] == "business":
    expected_tools.append(ToolCall(name="evidence.fetch_external_evidence",
                                   input_parameters={"case_id": CASE_ID,
                                                     "evidence_type": "registry_extract"}))
else:
    print("rule 4 not applicable: customer is not a business")

print()
print("EXPECTED TOOLS (derived, not observed)")
for tool in expected_tools:
    print(f"  {tool.name:<40} {json.dumps(tool.input_parameters)}")

In [ ]:
# --------------------------------------------------------------------------
# Validate the derived plan against the live tool catalogue.
#
# A golden naming a tool that does not exist, or passing an argument the tool's
# own schema rejects, is a broken golden - and would show up as an application
# failure. GET /api/mcp/tools returns each tool's JSON Schema straight from the
# MCP protocol, so this check cannot drift from the servers.
# --------------------------------------------------------------------------
from jsonschema import Draft202012Validator

mcp_tools = api("GET", "/api/mcp/tools", role="eval_reader")

SCHEMAS = {}
for server in mcp_tools:
    for tool in server["tools"]:
        SCHEMAS[tool["namespaced_name"]] = tool["input_schema"]

print("tools the application currently exposes:")
for name in sorted(SCHEMAS):
    print(f"  {name}")
print()

problems = []
for tool in expected_tools:
    if tool.name not in SCHEMAS:
        problems.append(f"{tool.name} is not exposed by any MCP server")
        continue
    errors = sorted(Draft202012Validator(SCHEMAS[tool.name]).iter_errors(
        tool.input_parameters), key=lambda e: e.path)
    for error in errors:
        problems.append(f"{tool.name} arguments rejected by its own schema: {error.message}")

if problems:
    raise RuntimeError(
        "The expected tool plan does not validate against the live tool catalogue:\n  "
        + "\n  ".join(problems)
        + "\nFix the golden - do not weaken the assertion."
    )
print("Every expected tool exists and its arguments validate against the tool's own "
      "JSON Schema.")

In [ ]:
# --------------------------------------------------------------------------
# Build the test case and print each DeepEval role explicitly.
# --------------------------------------------------------------------------
from deepeval.test_case import LLMTestCase

actual_tools = [
    ToolCall(name=entry["name"],
             input_parameters=entry["input_parameters"],
             output=entry["output"])
    for entry in export["tools_called"]
]

test_case = LLMTestCase(
    input=export["input"],
    actual_output=export["actual_output"],
    tools_called=actual_tools,
    expected_tools=expected_tools,
)

print("USER INPUT")
print(" ", test_case.input)
print()
print("EXPECTED TOOLS (derived from case facts + documented planner rules)")
for tool in test_case.expected_tools:
    print(f"  {tool.name:<40} {json.dumps(tool.input_parameters)}")
print()
print("ACTUAL TOOLS CALLED (observed via GET /api/eval/export/{run_id})")
for tool in test_case.tools_called:
    print(f"  {tool.name:<40} {json.dumps(tool.input_parameters)}")
print()
expected_names = sorted(t.name for t in test_case.expected_tools)
actual_names = sorted(t.name for t in test_case.tools_called)
print(f"names match (multiset): {expected_names == actual_names}")
print()
print("EXPECTED ARGUMENTS vs ACTUAL ARGUMENTS")
for tool in test_case.expected_tools:
    match = any(a.name == tool.name and a.input_parameters == tool.input_parameters
                for a in test_case.tools_called)
    print(f"  {'OK  ' if match else 'MISS'} {tool.name:<40} "
          f"{json.dumps(tool.input_parameters)}")

In [ ]:
# --------------------------------------------------------------------------
# Debug: what the planner said it would do, and what it deliberately skipped.
#
# GET /api/agent/trace/{run_id} exposes the planner's decision separately from
# execution, which is what makes a mismatch diagnosable: a missing tool that
# appears in skipped_tools with a stated reason is a planner decision, while a
# missing tool that appears in neither is a planning defect.
# --------------------------------------------------------------------------
trace = api("GET", f"/api/agent/trace/{RUN_ID}", role="eval_reader")

print(f"run status        : {trace['status']}")
print(f"tools_available   : {trace['tools_available']}")
print()
print("tools_selected (the planner's decision, before execution):")
for selected in trace["tools_selected"]:
    print(f"  {selected['server']}.{selected['tool']:<28} "
          f"args={json.dumps(selected['arguments'])}")
    print(f"      reason: {selected['reason']}")
print()
print("skipped_tools:")
for skipped in trace["skipped_tools"] or []:
    print(f"  {skipped['server']}.{skipped['tool']}: {skipped['reason']}")
if not trace["skipped_tools"]:
    print("  (none)")
print()
print("per-call status (a tool can be selected, called, and still fail):")
for call in trace["tool_calls"]:
    print(f"  {call['server']}.{call['tool']:<28} status={call['status']:<12} "
          f"latency_ms={call['latency_ms']}  error={call['error']}")

## Judge and threshold

- **No judge model scores this metric.** `ToolCorrectnessMetric` is a deterministic
  comparison: it makes no LLM call when measuring, costs nothing to score, and returns
  the same result every time for the same inputs.
- **`OPENAI_API_KEY` is nonetheless required.** In DeepEval 4.1.4 the constructor below
  builds a `GPTModel` regardless, and raises `DeepEvalError` without a key. The key is
  never used to produce the score.
- **Threshold**: `0.5`, DeepEval's documented default.

The metric is measured twice, because the two configurations answer different questions:

1. **Default `evaluation_params`** - compares tool *names* only. "Did the agent reach for
   the right tools?"
2. **`evaluation_params=[ToolCallParams.INPUT_PARAMETERS]`** - also compares arguments.
   "Did it call them with the right arguments?" This is the stricter and more useful gate
   here, because the planner's argument construction encodes real rules: a business
   customer's incorporation date must not be screened as a date of birth, and the country
   checked must be the beneficiary's.

`should_exact_match` and `should_consider_ordering` are left off. Ordering within the
tool-calling stage is not part of the published contract, and pinning it would create
failures on a change that harms nothing.

In [ ]:
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import ToolCallParams

metric = ToolCorrectnessMetric(
    threshold=0.5,                    # DeepEval's documented default
    include_reason=True,
    async_mode=False,
    verbose_mode=True,
    # ordering is not part of the published contract, so it is not pinned
    should_consider_ordering=False,
    should_exact_match=False,
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : none scores this metric - the comparison is deterministic")
print(f"               (DeepEval 4.1.4 still requires OPENAI_API_KEY to construct it)")
print(f"threshold    : {metric.threshold}")
print(f"eval params  : {metric.evaluation_params or '[names only]'}")

In [ ]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

In [ ]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

In [ ]:
# --------------------------------------------------------------------------
# Second, stricter measurement: arguments as well as names.
# --------------------------------------------------------------------------
strict_metric = ToolCorrectnessMetric(
    threshold=0.5,
    evaluation_params=[ToolCallParams.INPUT_PARAMETERS],
    include_reason=True,
    async_mode=False,
    verbose_mode=True,
    should_consider_ordering=False,
    should_exact_match=False,
)
strict_metric.measure(test_case)

print(f"metric      : ToolCorrectnessMetric (+ INPUT_PARAMETERS)")
print(f"threshold   : {strict_metric.threshold}")
print(f"score       : {strict_metric.score}")
print(f"PASS / FAIL : {'PASS' if strict_metric.is_successful() else 'FAIL'}")
print()
print("reason:")
print(textwrap.fill(str(strict_metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose log (debug) -----")
print(strict_metric.verbose_logs or "(none)")

## Limitations in a black-box acceptance test

1. **The golden encodes rules that could change without notice.** The planner's rules are
   documented but they are not part of the versioned response contract, so a rule change is
   a silent golden breakage. The schema validation cell catches a renamed *tool*; it cannot
   catch a changed *rule*. A failure here should always be read against `tools_selected`
   and `skipped_tools` in the trace before being called a regression.
2. **It is only valid on the first run for a case.** The planner skips tools that already
   succeeded for the case, so a second run legitimately calls fewer tools and this metric
   would report a false failure. Hence the reset requirement.
3. **Set comparison ignores duplicates in a way that matters here.** This scenario calls
   `screen_sanctions_pep` twice with different arguments. The name-only configuration
   cannot distinguish "screened both parties" from "screened one party twice", which is
   precisely why the `INPUT_PARAMETERS` measurement is run as well.
4. **Calling a tool is not using it.** The metric scores that a call was made. A tool that
   returned `status: "error"`, `"timeout"` or `"unavailable"` still counts as called - the
   debug cell prints per-call status for exactly that reason.
5. **Tool selection quality is not assessed here.** DeepEval can score selection against
   an `available_tools` catalogue, but that path invokes a judge and is a different
   question ("was this the best tool?") from the one this metric answers ("was it the
   expected tool?"). `ToolUseMetric` covers the former.